![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [125]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

# Load the dataset
df = pd.read_csv("cc_approvals.data", header=None) 
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,g,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,g,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,g,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,g,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,s,0,+


## 1. Cleaning Data 

In [126]:
# Check for missing values represented by '?'
condition = (df == '?').any(axis=1)
matching_rows = df[condition]
print("Rows with missing values:", matching_rows)

print("Number of rows with missing values:", len(matching_rows))

Rows with missing values:     0      1       2  3  4   5   6      7  8  9   10 11    12 13
83   a      ?   3.500  u  g   d   v  3.000  t  f   0  g     0  -
86   b      ?   0.375  u  g   d   v  0.875  t  f   0  s     0  -
92   b      ?   5.000  y  p  aa   v  8.500  t  f   0  g     0  -
97   b      ?   0.500  u  g   c  bb  0.835  t  f   0  s     0  -
206  a  71.58   0.000  ?  ?   ?   ?  0.000  f  f   0  p     0  +
248  ?  24.50  12.750  u  g   c  bb  4.750  t  t   2  g   444  +
254  b      ?   0.625  u  g   k   v  0.250  f  f   0  g  2010  -
270  b  37.58   0.000  ?  ?   ?   ?  0.000  f  f   0  p     0  +
286  a      ?   1.500  u  g  ff  ff  0.000  f  t   2  g   105  -
327  ?  40.83   3.500  u  g   i  bb  0.500  f  f   0  s     0  -
329  b      ?   4.000  y  p   i   v  0.085  f  f   0  g     0  -
330  b  20.42   0.000  ?  ?   ?   ?  0.000  f  f   0  p     0  -
346  ?  32.25   1.500  u  g   c   v  0.250  f  f   0  g   122  -
374  ?  28.17   0.585  u  g  aa   v  0.040  f  f   0  g  1004  -

In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       690 non-null    object 
 1   1       690 non-null    object 
 2   2       690 non-null    float64
 3   3       690 non-null    object 
 4   4       690 non-null    object 
 5   5       690 non-null    object 
 6   6       690 non-null    object 
 7   7       690 non-null    float64
 8   8       690 non-null    object 
 9   9       690 non-null    object 
 10  10      690 non-null    int64  
 11  11      690 non-null    object 
 12  12      690 non-null    int64  
 13  13      690 non-null    object 
dtypes: float64(2), int64(2), object(10)
memory usage: 75.6+ KB


In [128]:
df = df.replace('?', np.nan)
print(df.isnull().sum())

0     12
1     12
2      0
3      6
4      6
5      9
6      9
7      0
8      0
9      0
10     0
11     0
12     0
13     0
dtype: int64


In [129]:
df = df.dropna()

print(df.isna().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
dtype: int64


In [130]:
last_column = df.columns[-1]
df[last_column] = df[last_column].apply(lambda x:1 if x == '+' else 0)
df[last_column].value_counts()

13
0    361
1    298
Name: count, dtype: int64

## 2. Modeling

In [140]:
# Split the dataset into features and target variable
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [147]:
# Create dummy variables features
X_dummy = pd.get_dummies(X, drop_first=True)
X_dummy.columns = X_dummy.columns.astype(str)

In [148]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_dummy, y, test_size=0.2, random_state=42)

In [149]:
standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

In [150]:
logreg = LogisticRegression(max_iter=1000)
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}
grid_model = GridSearchCV(logreg, param_grid, cv=5)

grid_model.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear']})

In [155]:
best_model = grid_model.best_estimator_
y_pred = best_model.predict(X_test_scaled)
best_params = grid_model.best_params_
best_score = grid_model.best_score_

print("Best Parameters:", best_params)
print("Best Cross-Validation Score:", best_score)

Best Parameters: {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}
Best Cross-Validation Score: 0.8709613656783468
